# SAHA-AL Benchmark — Remaining Evaluation (Steps 4–9)

**Pre-requisites done locally (Steps 0–3):**
- Dataset stats
- Rule-based baselines (regex, spaCy, Presidio)
- Task 2 anonymization evaluation for all 8 systems

**This notebook runs (Steps 4–9):**
| Step | What | GPU? | Time |
|------|------|------|------|
| 4 | Task 1: Detection (span P/R/F1) | No | ~10s |
| 5 | Task 3: Privacy (CRR-3, ERA, UAC) | Yes | ~20-30 min |
| 6 | Bootstrap CIs | No | ~2 min |
| 7 | Failure Taxonomy | No | ~1 min |
| 8 | Pareto Frontier + plot | No | ~5s |
| 9 | BERT NER training | Yes (optional) | ~1-2h |

## Cell 0: Setup

In [ ]:
!pip install -q sentence-transformers faker bert_score spacy presidio-analyzer presidio-anonymizer
!python -m spacy download en_core_web_lg -q

In [ ]:
!git clone https://github.com/hanara2112/pii_identification_and_replacement.git repo

import os
os.chdir('repo/benchmark')
os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)
!pwd
!ls data/ predictions/

## Step 4: Task 1 — PII Detection Evaluation

Computes span-level P/R/F1 (exact, partial, type-aware) for rule-based baselines that have `_spans.jsonl` files.

In [ ]:
for mode in ['regex', 'spacy', 'presidio']:
    print(f'\n{"="*60}')
    print(f'  Detection eval: {mode}')
    print(f'{"="*60}')
    !python -m eval.eval_detection \
        --gold data/test.jsonl \
        --pred predictions/{mode}_spans.jsonl \
        --output results/eval_det_{mode}.json

### Interpret Step 4

- **Exact F1**: both start AND end must match → strictest
- **Partial F1**: character overlap IoU > 0.5 → catches boundary errors
- **Type-aware F1**: exact span + type must match → tests if system knows WHAT the PII is
- **Gap between Partial and Exact** = boundary error rate
- **Gap between Exact and Type-aware** = type confusion rate

## Step 5: Task 3 — Privacy Risk Assessment

Runs CRR-3, ERA (needs GPU for sentence-transformers), and UAC on key systems.

In [ ]:
# CRR-3 + ERA + UAC for the best seq2seq model
!python -m eval.eval_privacy \
    --gold data/test.jsonl \
    --pred predictions/predictions_bart-base-pii.jsonl \
    --train data/train.jsonl \
    --output results/eval_privacy_bart.json \
    --era-sample 500 \
    --skip-lrr

In [ ]:
# Privacy eval for flan-t5
!python -m eval.eval_privacy \
    --gold data/test.jsonl \
    --pred predictions/predictions_flan-t5-small-pii.jsonl \
    --train data/train.jsonl \
    --output results/eval_privacy_flan_t5.json \
    --era-sample 500 \
    --skip-lrr

In [ ]:
# Privacy eval for spaCy baseline
!python -m eval.eval_privacy \
    --gold data/test.jsonl \
    --pred predictions/spacy_predictions.jsonl \
    --train data/train.jsonl \
    --output results/eval_privacy_spacy.json \
    --era-sample 500 \
    --skip-lrr

In [ ]:
# Privacy eval for Presidio baseline
!python -m eval.eval_privacy \
    --gold data/test.jsonl \
    --pred predictions/presidio_predictions.jsonl \
    --train data/train.jsonl \
    --output results/eval_privacy_presidio.json \
    --era-sample 500 \
    --skip-lrr

### Interpret Step 5

- **CRR-3 ↓** (Capitalized 3-gram Re-id Rate): % of original capitalized 3-grams surviving in output. Lower = better privacy.
  - Expect: BART < 5%, spaCy ~20-30%, regex ~70%+
- **ERA@1 / ERA@5 ↓** (Entity Recovery Attack): Can a retrieval adversary guess the original entity from the anonymized text using sentence embeddings?
  - Random baseline = 1/pool_size. If ERA@1 >> random, the model is leaking semantic information.
- **UAC ↓** (Unique Attribute Combination): % of records with unique surviving attribute combos → re-identifiable.
  - Relates to k-anonymity. High UAC = quasi-identifiers still expose identity.

## Step 5b (Optional): LRR — LLM Re-identification Attack

Requires OpenAI API key. Costs ~$0.50–1.00 for 200 samples.

In [ ]:
# Uncomment and set your API key to run LRR
# import os
# os.environ['OPENAI_API_KEY'] = 'sk-...'  # or use Kaggle Secrets
#
# !python -m eval.eval_privacy \
#     --gold data/test.jsonl \
#     --pred predictions/predictions_bart-base-pii.jsonl \
#     --train data/train.jsonl \
#     --output results/eval_privacy_bart_with_lrr.json \
#     --era-sample 500 \
#     --lrr-sample 200

## Step 6: Bootstrap Confidence Intervals

In [ ]:
import json
import random
import re
import numpy as np

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def exact_entity_match(ent_text, pred_text):
    if not ent_text or not pred_text:
        return False
    return re.search(rf'(?<!\w){re.escape(ent_text)}(?!\w)', pred_text, re.IGNORECASE) is not None

def elr_fn(gold, preds):
    leaked, total = 0, 0
    for g, p in zip(gold, preds):
        pt = (p.get('anonymized_text') or '').strip()
        for ent in g.get('entities', []):
            total += 1
            if exact_entity_match(ent.get('text', ''), pt):
                leaked += 1
    return (leaked / total * 100) if total else 0

def bootstrap_ci(gold, preds, metric_fn, n_boot=1000, seed=42):
    rng = random.Random(seed)
    n = len(gold)
    scores = []
    for _ in range(n_boot):
        idx = [rng.randint(0, n-1) for _ in range(n)]
        scores.append(metric_fn([gold[i] for i in idx], [preds[i] for i in idx]))
    scores = np.array(scores)
    return {
        'mean': round(float(scores.mean()), 3),
        'ci_lower': round(float(np.percentile(scores, 2.5)), 3),
        'ci_upper': round(float(np.percentile(scores, 97.5)), 3),
    }

gold_test = load_jsonl('data/test.jsonl')
ci_results = {}

for name, pf in [('bart-base', 'predictions_bart-base-pii.jsonl'),
                  ('flan-t5-small', 'predictions_flan-t5-small-pii.jsonl'),
                  ('spacy', 'spacy_predictions.jsonl'),
                  ('presidio', 'presidio_predictions.jsonl')]:
    preds = load_jsonl(f'predictions/{pf}')
    pm = {p['id']: p for p in preds}
    ag = [g for g in gold_test if g['id'] in pm]
    ap = [pm[g['id']] for g in ag]
    ci = bootstrap_ci(ag, ap, elr_fn)
    ci_results[name] = ci
    print(f'  {name:20s}  ELR = {ci["mean"]:.3f}%  [{ci["ci_lower"]:.3f}, {ci["ci_upper"]:.3f}]')

with open('results/bootstrap_cis.json', 'w') as f:
    json.dump(ci_results, f, indent=2)
print('\nSaved to results/bootstrap_cis.json')

### Interpret Step 6

- 95% CI = if you re-ran the benchmark on a different test sample, results would fall in this range 95% of the time.
- **Narrow CI** = result is reliable. **Wide CI** = high variance, less trustworthy.
- For paper: report as `ELR = 0.93% [0.72, 1.15]`
- If CIs of two models DON'T overlap → difference is statistically significant.

## Step 7: Failure Taxonomy

In [ ]:
systems = {
    'bart-base-pii': 'predictions/predictions_bart-base-pii.jsonl',
    'flan-t5-small': 'predictions/predictions_flan-t5-small-pii.jsonl',
    't5-small': 'predictions/predictions_t5-small-pii.jsonl',
    'spacy': 'predictions/spacy_predictions.jsonl',
    'presidio': 'predictions/presidio_predictions.jsonl',
    'regex': 'predictions/regex_predictions.jsonl',
}

for name, pred_path in systems.items():
    print(f'\n{"="*60}')
    print(f'  Failure Taxonomy: {name}')
    print(f'{"="*60}')
    !python -m analysis.failure_taxonomy \
        --gold data/test.jsonl \
        --pred {pred_path} \
        --output results/failure_{name}.json

### Interpret Step 7

5 failure categories:

| Category | What it means | Implication |
|----------|--------------|-------------|
| **Full Leak** | Original entity appears verbatim in output | Model missed the entity entirely |
| **Boundary Error** | Part of entity leaked (e.g., first name kept, last name removed) | Detection is off by a few chars |
| **Type Confusion** | Replacement wrong format (e.g., replacing phone with a name) | Model confused PII types |
| **Ghost Leak** | Entity removed but surrounding context still reveals identity | Contextual information leakage |
| **Format Break** | Replacement doesn't preserve format (email → random text) | Utility issue |
| **Over-Masking** | Non-PII content was altered | False positive detection |

For the paper, make a stacked bar chart showing failure distribution across systems.

## Step 8: Pareto Frontier

In [ ]:
import json

# Collect all ELR + BERTScore results from local eval_anon_*.json files
all_eval_results = {}
eval_files = {
    'regex':         'Results/eval_anon_regex.json',
    'spacy':         'Results/eval_anon_spacy.json',
    'presidio':      'Results/eval_anon_presidio.json',
    'bart-base':     'Results/eval_anon_bart-base-pii.json',
    'flan-t5-small': 'Results/eval_anon_flan-t5-small-pii.json',
    'distilbart':    'Results/eval_anon_distilbart-pii.json',
    't5-small':      'Results/eval_anon_t5-small-pii.json',
    't5-eff-tiny':   'Results/eval_anon_t5-efficient-tiny-pii.json',
}

for name, path in eval_files.items():
    with open(path) as f:
        data = json.load(f)
    all_eval_results[name] = {
        'elr': data.get('elr_pct', data.get('elr', 0)),
        'bertscore': data.get('bertscore_f1', data.get('bertscore', 0)),
    }

with open('results/all_eval_results.json', 'w') as f:
    json.dump(all_eval_results, f, indent=2)

print('Collected results:')
for name, r in all_eval_results.items():
    print(f'  {name:20s}  ELR={r["elr"]:6.2f}%  BERTScore={r["bertscore"]:5.2f}')

In [ ]:
!python -m analysis.pareto_frontier \
    --results results/all_eval_results.json \
    --output results/pareto_analysis.json \
    --plot figures/pareto_frontier.png

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists('figures/pareto_frontier.png'):
    display(Image('figures/pareto_frontier.png'))
else:
    print('Plot not generated — check matplotlib availability')

### Interpret Step 8

- **X-axis** = Privacy (1 − ELR): 1.0 = perfect privacy, 0.0 = full leakage
- **Y-axis** = Utility (BERTScore/100): 1.0 = identical semantics, 0.0 = unreadable
- **Pareto-optimal** (red stars) = systems where no other system is better on BOTH axes
- **PUS(λ)** = combined score at different privacy-utility tradeoff points:
  - λ=0 → pure utility (regex wins)
  - λ=1 → pure privacy (BART wins)
  - λ=0.5 → balanced (likely BART or Flan-T5)

## Step 9 (Optional): BERT NER Training

Trains BERT-base-cased for token classification on SAHA-AL training data.
Needs GPU. Takes ~1-2 hours with T4.

In [ ]:
TRAIN_BERT = False  # Set to True to run

if TRAIN_BERT:
    # Step 1: Train
    !python -m baselines.bert_ner_baseline \
        --train data/train.jsonl \
        --val data/validation.jsonl \
        --output-dir checkpoints/bert_ner \
        --epochs 5 \
        --batch-size 32

    # Step 2: Predict
    !python -m baselines.bert_ner_baseline \
        --predict data/test.jsonl \
        --model-dir checkpoints/bert_ner/best_model \
        --output predictions/bert_ner_predictions.jsonl \
        --save-spans
else:
    print('Set TRAIN_BERT = True to run BERT NER training (needs GPU, ~1-2h)')

In [ ]:
# After BERT NER training, run detection eval on its output
import os
span_file = 'predictions/bert_ner_predictions_spans.jsonl'
if TRAIN_BERT and os.path.exists(span_file):
    print('Running detection eval on BERT NER predictions...')
    !python -m eval.eval_detection \
        --gold data/test.jsonl \
        --pred {span_file} \
        --output results/eval_det_bert_ner.json
elif TRAIN_BERT:
    print(f'Spans file not found at {span_file}. Check bert_ner output.')

## Summary: Collect All Results

In [ ]:
import json, os

print('=' * 70)
print('  SAHA-AL Benchmark — Complete Results')
print('=' * 70)

for d in ['Results', 'results', 'figures']:
    if os.path.exists(d):
        print(f'\n  {d}/:')
        for f in sorted(os.listdir(d)):
            fpath = os.path.join(d, f)
            size = os.path.getsize(fpath)
            print(f'    {f:45s} ({size:,} bytes)')

# Define eval files inline so this cell is self-contained
_eval_files = {
    'regex':         'Results/eval_anon_regex.json',
    'spacy':         'Results/eval_anon_spacy.json',
    'presidio':      'Results/eval_anon_presidio.json',
    'bart-base':     'Results/eval_anon_bart-base-pii.json',
    'flan-t5-small': 'Results/eval_anon_flan-t5-small-pii.json',
    'distilbart':    'Results/eval_anon_distilbart-pii.json',
    't5-small':      'Results/eval_anon_t5-small-pii.json',
    't5-eff-tiny':   'Results/eval_anon_t5-efficient-tiny-pii.json',
}

print('\n' + '=' * 70)
print('  Combined Task 2 Results')
print('=' * 70)
print(f'{"System":20s} {"ELR↓":>8s} {"TokRec↑":>8s} {"OMR↓":>8s} {"FPR↑":>8s} {"BERT↑":>8s}')
print('-' * 70)

for name, path in sorted(_eval_files.items()):
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        print(f'{name:20s} {d.get("elr", 0):7.2f}% {d.get("token_recall", 0):7.2f}% '
              f'{d.get("over_masking_rate", 0):7.2f}% {d.get("format_preservation_rate", 0):7.2f}% {d.get("bertscore_f1", 0):7.2f}')
    else:
        print(f'{name:20s}  [file not found: {path}]')

# Privacy results
print('\n' + '=' * 70)
print('  Combined Task 3 Results (Privacy)')
print('=' * 70)
print(f'{"System":20s} {"CRR-3↓":>8s} {"ERA@1↓":>8s} {"ERA@5↓":>8s} {"UAC↓":>8s}')
print('-' * 70)

for name in ['bart', 'flan_t5', 'spacy', 'presidio']:
    path = f'results/eval_privacy_{name}.json'
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        era = d.get('era') or {}
        print(f'{name:20s} {d.get("crr3", 0):7.2f}% {era.get("era_top1", 0):7.2f}% '
              f'{era.get("era_top5", 0):7.2f}% {d.get("uac", 0):7.2f}%')

In [ ]:
# Download all results
!cd /kaggle/working && tar czf saha_al_results.tar.gz results/ figures/ 2>/dev/null
print('Download saha_al_results.tar.gz from /kaggle/working/')